# Intro

## Contents

- [Workflow](#workflow)
- [Strategies](#strategies)
  - [Strategy characteristics](#strategy-characteristics)
  - [Best algorithms for determining entry and exit points](#best-algorithms-for-determining-entry-and-exit-points)
  - [Entry and exit logic](#entry-and-exit-logic)
  - [Strategies that use level detector](#strategies-that-use-level-detector)
- [Engine](#engine)
  - [Data providers](#data-providers)
  - [Strategy classes](#strategy-classes)
  - [Fees calculation](#fees-calculation)
  - [Trade direction](#trade-direction)
  - [How does a strategy choose what direction to trade?](#how-does-a-strategy-choose-what-direction-to-trade)
  - [Market opening hours](#market-opening-hours)
- [Analysis](#analysis)
  - [Strategy notebook structure](#strategy-notebook-structure)
  - [Live mode](#live-mode)
- [Strategy configuration](#strategy-configuration)
- [Strategy evaluation](#strategy-evaluation)
  - [Backtesting](#backtesting)
  - [Theoretical ceiling](#theoretical-ceiling)
  - [Parameter sweep](#parameter-sweep)
  - [Grid search](#grid-search)
  - [Walk-forward](#walk-forward)
  - [Monte Carlo simulations](#monte-carlo-simulations)

## Workflow

1. Choose a strategy to test or run live.
2. In data_configurator.py choose: provider (Bybit or Yahoo), symbol, interval, timeframe.
3. In trade_configurator.py choose: initial balance, leverage, position size, risk, fees, direction, etc.
4. In strategy_configurator.py choose: per-strategy parameters + exit
5. Run the strategy notebook.

## Strategies

### Strategy characteristics

- Consequent logic:
    - One position at a time. Every entry must be followed by an exit before the next entry. No two entries in a row.
    - Stateful position manager → new entries are allowed only when no position is open (can enter a trade only when flat).
- Two modes:
    - Historical
    - Live (unknown data): \
      Incremental processing (polls Bybit every interval, decides only on the latest candle → no lookahead).
- Visualization:
    - Interactive Plotly candlestick chart with entry-exit markers.

### Best algorithms for determining entry and exit points

Three current 2025-2026 professional strategies for BTCUSDT perpetuals on Bybit (breakout, momentum, trend-following):
1. Swing Level Breakout (Price Action) \
Best for scalping (1m-5m) and intraday (15m-1h). Uses dynamic swing highs/lows (like the level detector).
2. EMA Crossover + RSI Filter \
Momentum strategy, excellent for scalping and intraday across all TFs. \
Filters false signals with RSI.
3. SuperTrend Trend-Following \
Pure trend strategy with built-in ATR trailing stop. \
Extremely popular in crypto perpetuals for all timeframes; maximizes profit by riding trends while cutting losses fast.

### Entry and exit logic

These 3 strategies use ATR(14) for dynamic stops/tolerance (adapts to volatility).

1. Swing Level Breakout (builds directly on level detector code)

- Detects swing highs (resistance) and lows (support) with configurable left/right window.
- Long Entry: Price closes above a significant resistance level + confirmation candle.
- Short Entry: Price closes below a significant support level + confirmation candle.
- Exit Long: Price closes below next support or ATR trailing stop hit.
- Exit Short: Price closes above next resistance or ATR trailing stop hit.
- Ideal for scalping/intraday when price respects liquidity levels.

2. EMA Crossover + RSI Filter

- Fast EMA (9) / Slow EMA (21) – standard for crypto.
- Long Entry: Fast EMA crosses above Slow EMA AND RSI(14) < 70 (not overbought).
- Short Entry: Fast EMA crosses below Slow EMA AND RSI(14) > 30 (not oversold).
- Exit: Reverse crossover OR price hits ATR-based trailing stop.
- Excellent filter reduces whipsaws in ranging markets.

3. SuperTrend (most popular in 2025-2026 perpetuals)

- Combines ATR volatility bands with trend direction.
- Long Entry: SuperTrend flips from red to green (price closes above upper band).
- Short Entry: SuperTrend flips from green to red (price closes below lower band).
- Exit: SuperTrend flips opposite OR the built-in ATR trailing stop is hit (the SuperTrend line itself acts as dynamic stop).
- Extremely clean, low-lag, and maximizes trend capture while protecting capital.

### Strategies that use level detector

Consumers: the level_* family (level_breakout, level_breakout_inv).

How the level is chosen/defined:
- Selection: by strategy name \
  StrategyName.LEVEL_BREAKOUT / LEVEL_BREAKOUT_INV, --strategy level_breakout[_inv].
- Configuration: \
  In strategy_configurator.py in LevelParams.
- Wiring:
    - LevelStrategyBase.prepare calls detect_levels(df, …) seeding three detector families (resistance, support, + pullback if level_use_pullback),
    - then stores (confirmation_idx, invalidated_at, price) tuples;
    - _active_levels(i) returns the causally-confirmed, not-yet-invalidated level prices at bar i (look-ahead free — a pivot at i confirms at i + pivot_window).


## Engine

### Data providers

1. Bybit: crypto

2. Yahoo finance: indices/commodities/futures

Usage:
- python -m engine --strategy supertrend --provider yahoo --symbol GC=F --interval D --candles 500
- in a notebook: load_data(DataSpec(provider="yahoo", symbol="GC=F", interval="D", num_candles=500))

The strategies and backtester are provider-blind.

Things to know:
- Intervals: Yahoo supports 1 5 15 30 60 D W M only; an unsupported interval (3/120/240/360/720) is rejected at load.
- History depth: Yahoo intraday is shallow; daily/EOD is deep.
- Param scale: strategy defaults are tuned for BTC. Any knob in absolute mode (e.g. level_delta with delta_mode="absolute", in quote points) is price-scale-specific — use atr or percent mode for cross-asset, or retune. ATR/percent knobs are already scale-free.
- Costs: fee_bps/slippage_bps default to Bybit's; edit ACTIVE_TRADE for the other venue.
- swing_ml is the one real exception: its Tier-3 order-flow features are Bybit-only, and its model is trained on BTC — running it on gold would feed a BTC-trained model foreign data. swing_ml needs retraining per instrument.

### Strategy classes

Separate mode classes:
- one signal detector
- one entry signal
- separate mode classes for different directions
- in '_inv' mode the same signal is traded in the opposite direction
- -> 1 signal --> several directions
- -> a direction flip, not different entry logic

3 entry modes for one strategy class:
1. regular
2. inverse
3. adaptive

Separate strategy classes:
- one signal detector
- different entry signals on the same detector
- -> several different signals --> several different strategies
- ->  different entries

### Fees calculation

__basis points (bps) vs percent (%)__

Bybit quotes fees in %, but the engine stores and computes everything in basis points (bps). \
These are the same quantity in different units — converting is just a unit swap, not a calculation:

| Unit | Value |
|------|-------|
| 1 bp | 0.01% |
| 100 bps | 1% |
| Bybit taker 0.04% | 4 bps |
| Bybit taker 0.055% | 5.5 bps |

Conversion: bps = percent × 100 \
So Bybit's 0.055% → multiply by 100 → store 5.5 bps.

__Why the engine uses bps__

P&L is computed in bps, so costs share the same unit and subtract cleanly. In PositionState.exit():

raw_bps = (price - entry) / entry * 10_000   # fraction → bps \
trade.pnl_bps = raw_bps - cost_bps           # same unit, plain subtraction

The × 10_000 is the fraction → bps conversion (×100 to get %, ×100 again to get bps). \
Using bps avoids tiny decimals like 0.0004 and keeps fees, slippage, and returns on one consistent scale.

### Trade direction

- core.py class Direction: has no option 'BOTH'
- TradeDirection does: there is an option 'BOTH'

1. Direction (in core.py) is the physical side of one concrete trade — a single position can't be long and short at once. \
It's stamped on every Trade and Signal, and several pieces of code branch on it assuming exactly one of two values:
- P&L in PositionState.exit() — (price − entry) for a long, (entry − price) for a short. A BOTH trade has no defined P&L formula.
- PositionState.update_peak() — high-water (max) for longs, low-water (min) for shorts. BOTH has no defined trailing-stop direction.
- state.enter(direction, …) opens one position with one side; chart labels read "Long Entry" / "Short Entry".

Adding BOTH here would be a category error — it's not a third side, it's the absence of a single side, and every two-way branch above would need an undefined third case.

2. BOTH lives on TradeDirection (in trade_configurator.py) instead, because that's a different kind of thing: a policy/permission over many trades ("which sides is this run allowed to take"). \
"Both" is meaningful as a permission set; it's meaningless as the sign of one fill.

__The two are orthogonal__

| | Direction (core.py) | TradeDirection (trade_configurator.py) |
|---|---|---|
| what | sign of one trade | allowed sides for the run |
| values | LONG / SHORT | LONG / SHORT / BOTH |
| set by | strategy, per signal (computed) | config, once (ACTIVE_TRADE / --direction) |
| BOTH meaningful? | no — a position has one side | yes — it's a filter over all signals |

So the gate (TradeDirection.BOTH) says "allow longs and shorts," while the strategy still emits each individual trade as a definite Direction.LONG or Direction.SHORT. \
Keeping Direction binary is what lets the P&L and trailing-stop math stay branch-clean.


### How does a strategy choose what direction to trade?

The strategy's own signal logic decides the sign.

A strategy chooses direction itself, per bar, in its on_bar() logic — there's no central direction picker. \
It evaluates its indicators and calls either state.enter(Direction.LONG, …) or state.enter(Direction.SHORT, …). \
The chosen side is hardcoded into each strategy's entry branches.

The inverse variant (_inv strategy) is a separate class that flips the same signal: enters the opposite side.

So choosing direction happens at the strategy layer in two ways: 
- which signal logic fires the sign, and 
- which variant (base vs _inv) you run

The TradeDirection gate (TradingConfig.direction) doesn't choose — it can only veto. \
The strategy still emits a definite Direction.LONG/SHORT; if the run's gate disallows that side, state.enter() returns None and the attempt is counted as a suppressed entry.

So the flow is: \
strategy.on_bar()  →  picks Direction.LONG/SHORT  →  state.enter(direction, …) \
                                                         ├─ side allowed?  → opens the trade \
                                                         └─ side gated/halted? → None (suppressed)

How to test how a strategy performs only in LONG

Set the direction gate to LONG — the gate keeps only the strategy's long entries and suppresses every short.

3 ways:
1. In a notebook (recommended — inline, no global edit):

from engine.backtester import Backtester \
from engine.strategy_configurator import params_for \
from engine.strategies import SuperTrendStrategy \
from engine.trade_configurator import TradingConfig, TradeDirection \
from engine.data_configurator import load_data, ACTIVE

df = load_data() \
result = Backtester( \
    SuperTrendStrategy(params_for("supertrend")), \
    symbol=ACTIVE.symbol, \
    trading_config=TradingConfig(direction=TradeDirection.LONG),   # ← long-only \
).run(df, interval=ACTIVE.interval) \
print(result.summary())

2. Project-wide: \
Edit the ACTIVE_TRADE block in trade_configurator.py → direction = TradeDirection.LONG, then every notebook (and the CLI) inherits it.

3. CLI: \
python -m engine --strategy supertrend --direction long

Compare all three sides in one loop:

for d in (TradeDirection.BOTH, TradeDirection.LONG, TradeDirection.SHORT): \
    r = Backtester(SuperTrendStrategy(params_for("supertrend")), symbol=ACTIVE.symbol, \
                   trading_config=TradingConfig(direction=d)).run(df, interval=ACTIVE.interval) \
    print(f"{d.value:5} | P&L {r.total_pnl_bps:+8.1f} bps | {r.total_trades} trades | {r.suppressed_entries} suppressed")

Two things to get right:
1. Use the base strategy, not the _inv variant. \
supertrend + direction=long = the strategy's long setups only — exactly what you want. \
supertrend_inv + direction=long tests something different (it disengages an inversion leg) and will print a warning.
2. Long-only is not just "the long trades from the both-direction run." \
Because the engine holds one position at a time, suppressing a short leaves the book flat, which frees it to take a later long the both-run was too busy (in a short) to take. \
So trade count and timing legitimately differ — that's the true long-only equity path, which is what you're after. \
Check result.suppressed_entries (now shown in summary()) to see how many shorts were dropped.

Conclusion:
- The strategy decides the side from its indicators
- The config can only allow or block that side, never change it
- If you want the opposite side of a setup: run the _inv variant — not a config flag
- If you want to test a strategy performance on long entries only:  set the direction gate to LONG (via trade_configurator.py)

### Market opening hours

The hook is_market_open() is implemented for YahooProvider:
- It returns closed on Saturday (UTC) all day and Sunday before ~22:00 UTC (around the CME Globex weekly reopen).
- It returns open on weekdays.
- It's deliberately coarse:
    - weekday = open
    - no per-product hours, daily maintenance break, or holiday calendar \
     (those are DST/exchange-specific and risk false mid-week closes).
- LiveEngine calls the hook each tick, so it gates — on weekends it logs "market closed" and sleeps instead of polling Yahoo.
- __Caveat:__ if pointed at a 24/7 ticker on Yahoo (e.g. BTC-USD) it would wrongly pause weekends. \
  But the Yahoo provider is meant for the non-crypto markets.

Bybit stays 24/7 (no hook → always open).

## Analysis

### Strategy notebook structure

- Intro
- Configuration
    - Setup
    - Automatic: comes from the 3 configurators
    - Manual: 4 override cells
    - Final configuration
- Strategy
    - Backtesting: backtest + theoretical ceiling + entry-exit points chart
    - Grid search: a single full-history grid sweep + heatmap
    - Walk-forward analysis: rolling strategy parameters sweep + folds table + OOS equity curve + OOS trades chart
    - Monte Carlo simulations: monte carlo on OOS trades + histograms
    - Live signals
- Inverse strategy
    - Backtesting / Grid search / Walk-forward analysis / Monte Carlo simulations/ Live signals
- Adaptive strategy
    - Backtesting / Grid search / Walk-forward analysis / Monte Carlo simulations/ Live signals

Note: \
If a notebook has several strategy modes in it (base, inverse, adaptive, etc.), each strategy section ends in a blocking engine.run() live cell. \
So a "Run All" stops at the base strategy's live loop.




Notebook tuning - uniform across all strategy notebooks:
trade parameters configuration
strategy parameters configuration
exit parameters configuration
sweep of strategy and exit parameters


### Live mode

- The notebook Live cell
    - = a paper-trading/signal preview of the exact config you backtest
    - the parameters can be configured either in strategy_configurator.py or manually in the notebook

- Production live from CLI
    - = a long-running production process
    - is launched from the shell: --mode live
    - does not see notebook overrides
    - to configure its parameters, edit the defaults in strategy_configurator.py

__Live signals:__
- Live mode runs the same strategy / config / costs as the backtest.
- It generates signals on a rolling candle window \
  It tells you when to enter / exit + a sound notification.
- It does not place orders. \
  There's no exchange API key, no order execution.
- State persists \
  If you stop and restart, it remembers whether you're in a position via its SQLite state file under data/live/. \
  So a restart recovers an open position.
- Circuit breaker \
  If Bybit is unreachable 10 times in a row (10 consecutive fetch failures), it stops automatically instead of spinning forever.
- Chart updates in place \
  Automatic: the chart refreshes in the browser every poll_seconds.

__Notes:__
- Browser sound for live signals needs one gesture: \
  clicking Run on the live cell unlocks autoplay + notification permission for the session — that's automatic, no action needed, \
  so the first beep is silent before you've interacted with the tab.
- Telegram channel is skipped unless both env vars are set: it needs credentials to activate.
- To actually execute trades automatically, you need to add authenticated Bybit order placement on top of the signal output.


## Strategy configuration

Configuration:
- sets data, strategy, exit and trade parameters once
- each strategy below is then evaluated with the same four sections

Manual Configuration:
- overrides on top of the automatic config
- each cell applies dataclasses.replace to one handle
- leave a dict empty (or EXIT_POLICY = None) to keep that dimension automatic

Everything after Configuration uses only df, SYMBOL, INTERVAL, STRATEGY_CONFIG, EXIT_POLICY, TRADING_CONFIG.

The config rides on the strategy object, and both runners just use the strategy, so:

- Edit a default in strategy_configurator.py → params_for("strategy") returns it → flows to notebook backtest, notebook live-signals cell, and CLI production live. Everywhere.
- Override manually in strategy_name.ipynb → that STRATEGY_CONFIG object is used by both the notebook's backtest cell and the notebook's live-signals cell. \
A manual notebook override does not reach CLI production live, because the CLI builds from params_for(name) + flags, not the notebook.

Configuration options:
1. Centralized \
Notebooks read the global ACTIVE (data), params_for(name) (signal knobs), ACTIVE_TRADE (trade params), and each strategy's assigned exit preset. \
To change anything: edit a *_configurator.py file, which mutates the project-wide default for every notebook at once. \
There is no way to tune one notebook in isolation.

Example for a new exit preset: \
In strategy_configurator.py: add the entry in PER_STRATEGY_EXIT. You can assign any preset name there. \
PER_STRATEGY_EXIT = { \
    "order_block": "atr_stop_rr2",         # was "structural" — now uses a different preset by default \
    ... \
}

2. Per notebook \
Each strategy notebook is a self-contained tuning surface for data, strategy, exits, and trade parameters, \
where local edits override the centralized defaults (and any default baked into BaseStrategy/exit_policy_for) \
without touching the configurators — while leaving the centralized path fully intact when the local edits are left blank.

Each notebook's Configuration module is split into two:
- Configuration: automatic
    - This is the centralized default.
    - Defines the canonical handles from the three configurators.
- Configuration: manual
    - Four override cells + one resolve cell.
    - Each override applies dataclasses.replace on top of the automatic value.
    - An empty overrides dict means that dimension stays automatic.
    - EXIT_POLICY = None means each strategy keeps its assigned default.

Downstream the notebook references only the handles df, SYMBOL, INTERVAL, STRAT_CONFIG, EXIT_POLICY, TRADING_CONFIG. \
So "automatic vs manual" is decided in one place and the rest of the notebook is unchanged either way.

Example for a new exit preset: \
Inject in the notebook: \
from engine.strategy_configurator import EXIT_PRESETS \
strat = OrderBlockStrategy(cfg, exit_policy=EXIT_PRESETS\["fixed_2pct_rr3"\]()) \
Backtester(strat, ...).run(df, interval=INTERVAL) \
Sweep across the whole menu: \
for name in EXIT_PRESETS: \
    strat = SuperTrendStrategy(cfg, exit_policy=EXIT_PRESETS\[name\]()) \
    ...



base_config=STRATEGY_CONFIG — supplies the non-swept knobs (e.g. atr_period, the RSI settings); the swept knobs come from GRID.

swept = the keys in GRID; fixed = every other field, taken from base_config (STRATEGY_CONFIG).

The Configuration chapter (Setup/Automatic/Manual) is where you configure the fixed trade/data/exit context that every sweep combo runs under.
The GRID cell is where you configure what's swept.

WF deliberately re-optimizes only the dimensions that are genuine, drifting signal choices — which is essentially the strategy grid. 
If you want to act on grid search's winner, copy those values into STRATEGY_OVERRIDES (or the grid) yourself — there's no automatic handoff.

Non-swept means "whatever you left out of GRID." You can make every strategy parameter sweepable; you just list it in the grid. 


## Strategy evaluation

Flow:
- Backtest: see if a strategy has an edge.
- Grid search: pick the parameters (exit, interval, risk) and the strategy region → fix those.
- Walk-forward with sweep built-in: confirm that re-tuning the strategy knobs holds up OOS inside that context.

What each contributes:
- Backtest: first filter → a quick sanity check + the raw edge.
- Grid search: tells you which exit or interval to use.
- Walk-forward/rolling: robustness + honesty. \
  Shows if the edge keeps working out-of-sample, across different market periods, and is stable over time, not a one-window fluke.

In short:
- A backtest = it can work, walk-forward = it keeps working.
- Grid search to choose but walk-forward to trust the choice.
- Trust a strategy only after the rolling test, not the single backtest.

### Backtesting

In backtester.py every trade is bucketed by its pnl_bps:
- win → pnl_bps > 0
- loss → pnl_bps < 0
- BE → pnl_bps == 0 (exactly zero)

BE is a break-even trade that, after costs, netted precisely nothing — typically an instant flip or a force-close at the entry price. \
It's rare with continuous prices. \
It counts in the denominator of win_rate (wins / total) but is neither a win nor a loss, so a pile of BEs drags win rate down.

pnl_bps is net of fees + slippage


- Backtest:
    - Run the strategy once over a fixed stretch of history and look at the result.
    - Tells you: could this have made money at all? A quick check that an edge could exist.
    - Easy to fool yourself — you can tune the parameters until that one window looks great (overfitting).
    - Since it's easy to overfit, don't trust the "best" params it hands you.

### Theoretical ceiling

  - The highest theoretical P&L (in bps) an asset can reach.
  - A perfect strategy that buys every bottom and sells every top, trading the optimal chain of signals, net of the round-trip cost per trade. 
  - Tells you: what's the most money a strategy could possibly have made on this exact price series, using look-ahead information?
  - Also provides a skill ratio = strategy's actual P&L/ theoretical ceiling = the share of the maximum achievable profit that was realized.
  - The skill ratio tells you: you captured so many % of what was theoretically possible.
  - For OHLCV-only crypto:
    - ~5–15% is realistic
    - below ~5% means no edge

### Parameter sweep

(was before grid search)

  - Sweeps strategy parameters only.
  - This standalone sweep is not feeding the walk-forward.
  - It runs over the full history and its result is never passed to walk_forward() — the two cells are independent.
  - Usage:
    - exploratory analysis to see the parameters landscape
    - sanity-check of the grid bounds
    - not for selecting the parameters you then "validate"
  


GRID = {"ema_fast": [5, 9, 13, 17], "ema_slow": [20, 30, 40, 50]} \
MIN_TRADES = 2          # ignore degenerate combos with too few trades to be meaningful \
sw = sweep(STRATEGY, df, GRID, symbol=SYMBOL, interval=INTERVAL, \
           trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY, \
           base_config=STRATEGY_CONFIG)   # non-swept knobs come from STRATEGY_CONFIG (automatic + manual) \
sw[sw.trades >= MIN_TRADES].sort_values("total_pnl_bps", ascending=False).head(8)

__Exit Parameters Sweep__

For sweeping: skip the presets and build the parameterized classes directly, injecting via exit_policy.

from engine.exits import ChandelierStop, CompositeExit, AtrStop, RrTarget \
for mult in (2.0, 2.5, 3.0): \
    strat = SuperTrendStrategy(cfg, exit_policy=ChandelierStop(mult)) \
    Backtester(strat, symbol=SYMBOL, trading_config=ACTIVE_TRADE).run(df, interval=INTERVAL)

Or a 2-D SL×RR sweep: \
for a in (1.0, 1.5, 2.0): \
    for rr in (1.5, 2.0, 3.0): \
        strat = SuperTrendStrategy(cfg, exit_policy=CompositeExit(AtrStop(a), RrTarget(rr))) \
        ...

### Grid search

- A single full-history grid run across all dimensions.
- Use it to choose the settings before running walk-forward validation:
    - which exit policy suits this strategy?
    - which interval/symbol does it work on?
    - what leverage for my risk?
    - which parameter ranges produce consistently good results?
- It's overfit-prone (in-sample best) — that's fine, it's for choosing, not validating.

__Heatmap__

HEATMAP_METRIC = is a configurable knob: flip between Sharpe / P&L / profit_factor without editing the plot.

- P&L shows where the money is \
  But: a combo can look "green" just because it took more trades, or because one big move dominated — \
  neither means a dependable region. Sharpe normalizes that out.
- Sharpe shows where it's reliable \
  Sharpe is risk-adjusted (mean trade return ÷ its volatility), so a smooth green patch means consistent performance. \
  Look at the region of good parameters, not a single peak: 
  - a bright cell with bright neighbours is robust,
  - a lone bright cell is usually an overfit outlier.

### Walk-forward

__Sweep__

- Belongs inside walk-forward, on the training slice only — never on the test slice (that's curve-fitting).
- Sweep params on each training slice, choose the best combo, score it on the next unseen slice, then slide forward and repeat.
  * If the full history is swept once in backtester and then walk-forwarded with those fixed params -> you get look-ahead bias: \
    the params are chosen with knowledge of the test windows, so "testing" on them proves nothing. \
    That leak is precisely what walk-forward is designed to eliminate.

__Walk-forward (rolling)__

- Catches overfitting / curve-fit parameters.
 - An out-of-sample validator for the one thing you adaptively re-tune over time — the strategy params — with everything else fixed.
- Repeat the test over many shifting windows: pick (or tune) on an earlier slice, then check on the next unseen slice, then slide forward and repeat.
- Stitching the out-of-sample (OOS) test trades together gives a look-ahead-free track record. If the edge only exists in-sample, it dies here.
- Tells you: does it keep working on data it wasn't tuned on, across different market periods? Are optimal parameters stable over time?
- No separate validation backtest is needed afterward: walk-forward already is out-of-sample backtesting, \
  so its stitched-together out-of-sample result is your real verdict. \
  A later full-history backtest is only for eyeballing the equity curve or refitting final params for live.

__Caveats:__
- Window sizing is itself a hyperparameter. \
  If results only look good at one specific TRAIN_BARS/TEST_BARS, it can be an overfit —> vary them.
- Block size trades off preserving serial correlation (larger) against resampling diversity (smaller). \
  block=5 is a reasonable default for trade sequences.



Walk-forward analysis:
- re-sweeps parameters inside each train window
— rolling out-of-sample validation:
    - each test window is traded with the parameters that won the preceding train window's sweep
    - then the OOS trades are stitched together
- WF efficiency (OOS/IS) ≈ 1 means the in-sample edge carried over; ≪ 1 (or n/a) means it was curve-fit

Walk-forward out-of-sample trades with entries/exits \
Every trade lives entirely inside one fold's window. \
So the walk-forward trades are not generated by one EMA pair — each fold used its own winning ema_fast/ema_slow. \
And when you overlay those trades on a single EMA(9/21):
- The markers are exact (real entry/exit prices and times — always correct).
- The EMA lines are exact (true historical EMA(9/21)).
- But the two don't explain each other. \
  An entry that fired because EMA(13) crossed EMA(34) in fold 2 will not sit on a cross of the drawn EMA(9/21). \
  Only trades from folds that happened to pick 9/21 will line up.

  

walk forward objective=total_pnl_bps

optimize on total_pnl_bps and not USD net profit, because it's sizing-invariant (size-independent) — and that's exactly what you want for parameter selection.

The walk-forward objective decides which parameter combo wins each in-sample window. You want to pick the combo with the best signal edge, not the one that happens to dance well with your sizing/leverage/compounding. total_pnl_bps is the pure per-trade return stream net of costs, independent of capital deployed.

optimize in bps (clean signal metric), report the final result in dollars (the OOS equity curve in [b-eq] and Monte Carlo use USD)


wf.summary()
The summary reports OOS in bps plus the unit-free WF efficiency ratio (ΣOOS/ΣIS).


param_stability()
It's sorted by fold (chronological) order.
That's the right order and it should not be sorted by parameter value: the entire point is to read down the time axis and see whether the chosen params jump around (jumpy = overfit-prone, stable = trustworthy). Sorting by value would destroy that reading.

If what you actually want is to quantify stability rather than eyeball it, don't sort — aggregate:
That tells you stability directly (e.g. ema_slow stuck at 50 every fold → nunique == 1).

wf.folds_frame()
= the same chosen params measured on two different windows
- is_total_pnl_bps — in-sample: the total bps P&L the winning combo scored on the training window. It's what the optimizer thought was best. (The column is literally named is_{objective}; with the default objective="total_pnl_bps" it's is_total_pnl_bps. Pick a different objective and you'd see e.g. is_sharpe_approx instead.)
- oos_total_pnl_bps — out-of-sample: what those exact params then delivered on the test window — counting only trades that entered inside [test_start, test_end]. This is the honest, look-ahead-free result.

Reading them side by side is the overfit check:
- if oos collapses far below is (or flips negative) the params didn't generalize. 
- The summary aggregates this into WF efficiency = Σoos / Σis (≈1 robust, ≪1 overfit, <0 inverted).

The default is to sweep only a couple of params -> 3 reasons:
- Cost \
  A grid is a Cartesian product. \
  With the number of parameters equal to: 3·4·4·2·2·3·3 = 1,728 combos. \
  In grid_search that's 1,728 backtests; in walk_forward it re-sweeps every fold, so 1,728 × 5 folds ≈ 8,640 backtests — vs 16/fold when you sweep just ema_fast/ema_slow.
- Overfitting \
  Walk-forward exists to measure honest out-of-sample edge. \
  Every extra dimension you optimize in-sample fits more noise, so wf.param_stability() gets jumpier and OOS results get less trustworthy. \
  Fewer swept knobs → more robust readout. \
  It's still look-ahead-free either way — just noisier.
- base_config supplies the rest \
  The omitted fields aren't unset — they come from STRATEGY_CONFIG (Automatic + Manual config), \
  which is exactly how the notebook keeps the search focused on the knobs you're testing while holding your chosen context constant.


### Monte Carlo simulations

- Catches luck / sequence risks.
- It is run on the walk-forward OOS trades, never on the sweep winner.
- It estimates the range of possible final returns and max drawdowns.
- Tells you: given these trades, what's the distribution of drawdown / terminal equity? Is the edge distinguishable from zero?
- P(profitable) near 50% means the OOS edge is indistinguishable from noise (close to a random chance).

__WF vs Monte Carlo:__
- By resampling consecutive trades instead of randomly shuffling them, we preserve losing streaks and market regimes; \
  random shuffling tends to underestimate drawdowns.
- The walk-forward gave us ONE ordering of the out-of-sample trades. \
  In Monte Carlo we shuffle those trades thousands of times and rebuild the equity curve each time. \
  The shuffle is done in small contiguous chunks, so losing streaks stay realistic.

__Caveats:__
- Monte Carlo assumes the OOS trades are representative.
- It quantifies luck given the edge; it cannot rescue a non-edge. That's why it runs on the walk-forward OOS trades, not the sweep winner.